# Milestone 1 Exploration

In [243]:
import pandas as pd
import json

data = pd.read_parquet("../data/raw/reviews.parquet")
metadata = pd.read_parquet("../data/raw/metadata.parquet")

## Dataset Overview

### User Reviews

In [244]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4880181 entries, 0 to 4880180
Data columns (total 10 columns):
 #   Column             Dtype  
---  ------             -----  
 0   rating             float64
 1   title              str    
 2   text               str    
 3   images             object 
 4   asin               str    
 5   parent_asin        str    
 6   user_id            str    
 7   timestamp          int64  
 8   helpful_vote       int64  
 9   verified_purchase  bool   
dtypes: bool(1), float64(1), int64(2), object(1), str(5)
memory usage: 1.3+ GB


In [245]:
data.head(1000).describe(include="all")

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
count,1000.000000,1000,1000,1000,1000,1000,1000,1.000000e+03,1000.000000,1000
unique,NaN,722,866,1000,802,801,240,NaN,NaN,2
top,NaN,Cool,Cool,[],B07T771SPH,B07T771SPH,AGQXWQL42ONKPE4ADQF565EJSV2Q,NaN,NaN,True
freq,NaN,52,46,1,12,12,79,NaN,NaN,885
mean,3.721000,NaN,NaN,NaN,NaN,NaN,NaN,1.501493e+12,5.302000,NaN
std,1.446808,NaN,NaN,NaN,NaN,NaN,NaN,1.062314e+11,19.209628,NaN
min,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,9.764227e+11,0.000000,NaN
25%,3.000000,NaN,NaN,NaN,NaN,NaN,NaN,1.428865e+12,0.000000,NaN
50%,4.000000,NaN,NaN,NaN,NaN,NaN,NaN,1.518549e+12,1.000000,NaN
75%,5.000000,NaN,NaN,NaN,NaN,NaN,NaN,1.593010e+12,4.000000,NaN


In [246]:
data.isna().sum()

rating               0
title                0
text                 0
images               0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0
dtype: int64

In [247]:
data.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,1.0,malware,mcaffee IS malware,[],B07BFS3G7P,B0BQSK9QCF,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1562182632076,0,False
1,5.0,Lots of Fun,I love playing tapped out because it is fun to...,[],B00CTQ6SIG,B00CTQ6SIG,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,1424120336000,0,True
2,5.0,Light Up The Dark,I love this flashlight app! It really illumin...,[],B0066WJLU6,B0066WJLU6,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,1362399267000,0,True
3,4.0,Fun game,One of my favorite games,[],B00KCYMAWK,B00KCYMAWK,AH6CATODIVPVUOJEWHRSRCSKAOHA,1561061428662,0,True
4,4.0,I am not that good at it but my kids are,Cute game. I am not that good at it but my kid...,[],B00P1RK566,B00P1RK566,AEINY4XOINMMJCK5GZ3M6MMHBN6A,1418257196000,0,True


In [248]:
data.shape

(4880181, 10)

#### Field Selection

| Field | Type | Explanation | Keep? | Justification |
|---|---|---|---|---|
| rating | float | Rating of the product (from 1.0 to 5.0). | No | Does not effectively contribute to ranking beyond what can already be done using `average_rating` in item metadata. |
| title | str | Title of the user review. | Yes | Can contain useful information about the product. |
| text | str | Text body of the user review. | Yes | Can contain useful information about the product. |
| images | list | Images that users post after they have received the product. Each image has different sizes (small, medium, large), represented by the small_image_url, medium_image_url, and large_image_url respectively. | No | Difficult to extract information from and relatively rare. |
| asin | str | ID of the product; unique for each colour/style/size of the same product. | No | Does not provide any parsable information. |
| parent_asin | str | Parent ID of the product. Note: Products with different colors, styles, sizes usually belong to the same parent ID. The “asin” in previous Amazon datasets is actually parent ID. Please use parent ID to find product meta. | Yes | Used as an identifier to aggregate project information and join results with item metadata. |
| user_id | str | ID of the reviewer | No | Does not provide any parsable information. |
| timestamp | int | Time of the review (unix time) | No | Not interested in considering recency of review when retrieving products. |
| verified_purchase | bool | User purchase verification | Yes | Used to filter out reviews without a verified purchase in order to ensure that review information is reliable; the column will be dropped afterwards. |
| helpful_vote | int | Helpful votes of the review | No | Could be helpful in weighing importance of each review, but difficult to implement in practice. Additionally, the vast majority of reviews will have 0 helpful votes. |

### Item Metadata

In [249]:
metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 89251 entries, 0 to 89250
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    87482 non-null  str    
 1   title            89251 non-null  str    
 2   average_rating   89226 non-null  float64
 3   rating_number    86292 non-null  float64
 4   features         89251 non-null  object 
 5   description      89251 non-null  object 
 6   price            89251 non-null  str    
 7   images           89251 non-null  object 
 8   videos           89251 non-null  object 
 9   store            89038 non-null  str    
 10  categories       89251 non-null  object 
 11  details          89251 non-null  str    
 12  parent_asin      89251 non-null  str    
 13  bought_together  0 non-null      str    
 14  subtitle         0 non-null      str    
 15  author           0 non-null      str    
dtypes: float64(2), object(5), str(9)
memory usage: 55.0+ MB


In [250]:
metadata.head(1000).describe(include="all")

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
count,1000,1000,1000.000000,588.000000,1000,1000,1000,1000,1000,998,1000,1000,1000,0,0,0
unique,2,998,NaN,NaN,1000,1000,52,1000,269,864,1000,999,1000,0,0,0
top,Appstore for Android,Slots,NaN,NaN,[All the pressing point has been explained wit...,[Acupressure technique is very ancient and ver...,0.0,"{'hi_res': [None, None, None, None], 'large': ...","{'title': [''], 'url': [''], 'user_id': ['']}",Microsoft,[],"{""Is Discontinued By Manufacturer"": ""No"", ""Ite...",B00VRPSGEO,NaN,NaN,NaN
freq,833,3,NaN,NaN,1,1,652,1,732,11,1,2,1,NaN,NaN,NaN
mean,NaN,NaN,3.386600,329.756803,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,0.784791,1655.969041,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,3.000000,5.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,3.400000,12.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,3.800000,59.250000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [251]:
metadata.isna().sum()

main_category       1769
title                  0
average_rating        25
rating_number       2959
features               0
description            0
price                  0
images                 0
videos                 0
store                213
categories             0
details                0
parent_asin            0
bought_together    89251
subtitle           89251
author             89251
dtype: int64

In [252]:
metadata.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Appstore for Android,Accupressure Guide,3.6,NaN,[All the pressing point has been explained wit...,[Acupressure technique is very ancient and ver...,0.0,"{'hi_res': [None, None, None, None], 'large': ...","{'title': [''], 'url': [''], 'user_id': ['']}",mAppsguru,[],"{""Release Date"": ""2015"", ""Date first listed on...",B00VRPSGEO,NaN,NaN,NaN
1,Appstore for Android,Ankylosaurus Fights Back - Smithsonian's Prehi...,4.0,NaN,[ENCOURAGE literacy skills with highlighted na...,[Join Ankylosaurus in this interactive book ap...,2.99,"{'hi_res': [None, None, None, None, None], 'la...","{'title': [''], 'url': [''], 'user_id': ['']}","Oceanhouse Media, Inc",[],"{""Release Date"": ""2014"", ""Date first listed on...",B00NWQXXHQ,NaN,NaN,NaN
2,Appstore for Android,Mahjong 2015,3.1,NaN,[Mahjong 2015 is a free solitaire matching gam...,[Mahjong 2015 is a free solitaire matching gam...,0.0,"{'hi_res': [None, None, None], 'large': ['http...","{'title': [''], 'url': [''], 'user_id': ['']}",sophiathach,[],"{""Release Date"": ""2014"", ""Date first listed on...",B00RFKP6AC,NaN,NaN,NaN
3,Appstore for Android,Jewels Brick Breakout,4.2,NaN,"[Game Features:, - Intuitive touch controls wi...",[Jewels Brick Breakout is a glowing jewels bri...,0.0,"{'hi_res': [None, None, None, None, None, None...","{'title': [''], 'url': [''], 'user_id': ['']}",Bad Chicken,[],"{""Release Date"": ""2015"", ""Date first listed on...",B00SP2QU0E,NaN,NaN,NaN
4,Appstore for Android,Traffic Police: Off-Road Cub,3.3,NaN,"[In this game you will find:, - Killer police ...",[Become the best road police officer in Cube C...,0.0,"{'hi_res': [None, None, None, None], 'large': ...","{'title': [''], 'url': [''], 'user_id': ['']}",Dast 2 For Metro,[],"{""Release Date"": ""2016"", ""Date first listed on...",B01DZIT64O,NaN,NaN,NaN


In [253]:
metadata.shape

(89251, 16)

#### Field Selection

| Field | Type | Explanation | Keep? | Justification |
|---|---|---|---|---|
| main_category | str | Main category (i.e., domain) of the product. | No | All software for the most part, so not helpful in the search. |
| title | str | Name of the product. | Yes | Can contain useful information about the product. |
| average_rating | float | Rating of the product shown on the product page. | Yes | Can be used to sort the products after retrieving them (in conjunction with `rating_number`). |
| rating_number | int | Number of ratings in the product. | Yes | Can be used to sort the products after retrieving them (in conjunction with `average_rating`). |
| features | list | Bullet-point format features of the product. | Yes | Can contain useful information about the product. |
| description | list | Description of the product. | Yes | Can contain useful information about the product. |
| price | float | Price in US dollars (at time of crawling). | No | Difficult to determine a relative price range for product that can be determined through a query. |
| images | list | Images of the product. Each image has different sizes (thumb, large, hi_res). The “variant” field shows the position of image. | No | Difficult to extract information from. |
| videos | list | Videos of the product including title and url. | No | Difficult to extract information from. |
| store | str | Store name of the product. | Yes | User may directly query for a certain store. |
| categories | list | Hierarchical categories of the product. | Yes | Can contain helpful information on the type of product. |
| details | dict | Product details, including materials, brand, sizes, etc. | Yes* | Include certain possible keys of dict: see below for more information. |
| parent_asin | str | Parent ID of the product. | Yes | Used as an identifier and to join results with user reviews. |
| bought_together | list | Recommended bundles from the websites. | No | All values are empty. |
| subtitle | str | No description provided in documentation. | No | All values are empty. |
| author | str | No description provided in documentation. | No | All values are empty. |

##### Details
The item metadata has column `details`, which contains dicts of various product details that vary from product to product. Due to the number of possible unique keys contained in each of the dictionaries, we will only list the ones that we chose to keep and incorporate into the item metadata dataframe. We generally chose to exclude details for one or more of the following reasons:

- A very small number (< 100) of items included the particular detail.
- The detail did not include any information that we found meaningful in the context of information retrieval.
- The values corresponding to the detail is structured or formatted in a way that is difficult to effectively parse.

The following table includes the details that we decided to retain:

| Detail | Type | Justification |
|---|---|---|
| Developed By | str | Users may query for a specific developer. |
| Version | str | Users may query for a specific version of the software. |
| Application Permissions | str | Users may query based on information related to application permissions. |
| Minimum Operating System | str | Users may query based on their desired operating system. |
| Manufacturer | str | Users may query based on manufacturer. |
| Language | str | Users may query based on available language for the software. |

## Preprocessing

### Pipeline overview

All preprocessing for the retrieval dataset is implemented in `src/preprocessing/clean_data.py`, with column lists and paths in `src/preprocessing/constants.py` and reusable SQL-style transforms in `src/preprocessing/helpers.py`. Processing uses **DuckDB** on Parquet so we can filter, aggregate, and join a large review table without loading everything into pandas at once. The script writes a single table to `data/processed/preprocessed_data.parquet`.

The pipeline has three stages: **clean reviews**, **clean metadata**, and **merge**. The merge is an **inner join** on `parent_asin`, so a row exists only when both a metadata record and at least one verified review (after aggregation) exist for that parent product.

---

### 1. Reviews (`clean_reviews`)

1. **Read Parquet** — `read_parquet` loads only `title`, `text`, `parent_asin`, and `verified_purchase` from `data/raw/reviews.parquet` (see `REVIEWS["READ"]`). Other raw columns are never read, which keeps I/O and memory bounded. 

   **Columns omitted at read time (and why):** 
   - `rating`: redundant for retrieval given `average_rating` in metadata and not used for ranking in this pipeline
   - `images`: hard to use for text retrieval
   - `asin` and `user_id`: identifiers with no usable text for matching
   - `timestamp`: recency is out of scope
   - `helpful_vote`: could weight reviews but is sparse.

2. **Filter verified purchases** — `filter_by_column` keeps rows with `verified_purchase = True`. This matches the exploratory decision to prioritize trustworthy reviews. The flag column is dropped later and does not appear in the final dataset.

3. **Concatenate title and text per review** — `concat_columns` builds `reviews_content` by joining `title` and `text` with `concat_ws` (default separator: a single space). Parts are cast to `VARCHAR`. Null pieces are skipped by `concat_ws`, so missing title or text does not break the string. The original `title` and `text` columns are removed (`EXCLUDE`) once merged into `reviews_content`.

4. **One row per product** — `concat_rows` groups by `parent_asin` and aggregates all `reviews_content` strings for that product with `string_agg(..., ' ')` (space-separated). 

   **Rationale:** many reviews share the same `parent_asin`. Retrieval and indexing are at product level, so we collapse multiple review rows into one searchable text blob per product.

5. **Final review columns** — `select_columns` keeps `parent_asin` and `reviews_content` only.

---

### 2. Metadata (`clean_metadata`)

1. **Read Parquet** — `read_parquet` selects the columns listed in `METADATA["READ"]`: `title`, `average_rating`, `rating_number`, `features`, `description`, `store`, `categories`, `details`, and `parent_asin`. Other raw columns are never read, which keeps I/O and memory bounded. 

   **Columns omitted at read time (and why):** 
   - `main_category`—The slice is largely one domain (e.g. Android apps), so it adds little discriminative text for search
   - `price`—hard to interpret 
   - `images` and `videos`—Same as above
   - `bought_together`, `subtitle`, and `author`—empty in this dataset for all rows, so they carry no signal.

2. **Duplicate `title` as `product`** — `duplicate_column` adds `product` as a copy of `title`. The exploratory notes treat `title` as the product name for search. We keep a dedicated `product` column for downstream use while still using `title` inside the large metadata text field (see concat step below).

3. **Flatten list-like columns** — `collapse_array_to_string` converts `categories`, `features`, and `description` from lists to a single string each, using `array_to_string(..., ', ')`.

4. **Missing numeric aggregates** — `convert_nan_to_negative_one` sets `average_rating` and `rating_number` to **-1** when the value is NULL or NaN (`isnan`). -1 is an explicit placeholder that downstream code can filter or interpret (e.g., “unknown” vs a real 1–5 star average).

5. **Parse `details` as JSON** — `convert_string_to_json` casts `details` to DuckDB’s `JSON` type so keys can be extracted reliably.

6. **Expand selected JSON keys into columns** — `expand_json_to_columns` drops the original `details` column and materializes these keys as separate text columns:  
   `Developed By`, `Version`, `Application Permissions`, `Minimum Operating System`, `Manufacturer`, `Language`.  
   Extraction uses `json_extract` with JSON paths that quote keys (including spaces). Values are normalized: empty/missing becomes `''`; string values are trimmed of surrounding quotes, values that look like JSON arrays are flattened into comma-separated text.

7. **Single metadata text field** — `concat_columns` merges, in order:  
   `developed_by`, `version`, `application_permissions`, `minimum_operating_system`, `manufacturer`, `language`, `features`, `description`, `store`, `categories`, `title`  
   into one column `metadata_content` (space-separated, same `concat_ws` behavior as reviews).

8. **Final metadata columns** — `select_columns` keeps `parent_asin`, `product`, `average_rating`, `rating_number`, and `metadata_content`.

---

### 3. Merge (`merge_reviews_and_metadata`)

1. **Join** — `INNER JOIN ... USING (parent_asin)` combines the aggregated reviews relation with metadata. Products with no reviews (after verification) or no metadata row are excluded.

2. **Unified document text** — `concat_columns(joined, ["reviews_content", "metadata_content"])` appends review text and metadata text into **`data_content`** (the default name in `concat_columns` when not overridden). Review signal and catalog signal are therefore in one field for retrieval. `reviews_content` and `metadata_content` are removed from the relation as part of that operation.